In [1]:
import rasterio
from rasterio.sample import sample_gen
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd


In [8]:
# 📁 Шлях до LULC
lulc_root = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/lulc")

In [3]:
# 🗂️ LULC словник
lulc_classes_colors = {
    0:  {"name": "No Data", "color": "#FFFFFF"},
    1:  {"name": "Water", "color": "#1A5BAB"},
    2:  {"name": "Trees", "color": "#358221"},
    4:  {"name": "Flooded Vegetation", "color": "#87D19E"},
    5:  {"name": "Crops", "color": "#FFDB5C"},
    7:  {"name": "Built Area", "color": "#ED022A"},
    8:  {"name": "Bare Ground", "color": "#EDE9E4"},
    9:  {"name": "Snow/Ice", "color": "#F2FAFF"},
    10: {"name": "Clouds", "color": "#C8C8C8"},
    11: {"name": "Rangeland", "color": "#A7D282"}
}


In [4]:

# 📖 ICESat дані
ice_gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_32635.parquet")  # заміни на свій шлях


In [5]:
# Створюємо нову колонку з роком на основі індексу
ice_gdf["year"] = ice_gdf.index.year


In [6]:
ice_gdf.crs

<Projected CRS: EPSG:32635>
Name: WGS 84 / UTM zone 35N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 24°E and 30°E, northern hemisphere between equator and 84°N, onshore and offshore. Belarus. Bulgaria. Central African Republic. Democratic Republic of the Congo (Zaire). Egypt. Estonia. Finland. Greece. Latvia. Lesotho. Libya. Lithuania. Moldova. Norway. Poland. Romania. Russian Federation. Sudan. Svalbard. Türkiye (Turkey). Uganda. Ukraine.
- bounds: (24.0, 0.0, 30.0, 84.0)
Coordinate Operation:
- name: UTM zone 35N
- method: Transverse Mercator
Datum: World Geodetic System 1984
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [9]:
# 📦 Порожній список для збереження результатів
gdfs = []

# 🔁 Обробка по кожному року
for year in sorted(ice_gdf["year"].unique()):
    subset = ice_gdf[ice_gdf["year"] == year].copy()
    if subset.empty:
        continue

    # Пошук відповідного LULC-файлу
    candidates = [
        lulc_root / f"lulc_{year}.tif",
        lulc_root / f"lulc{year}.tif"
    ]
    lulc_path = next((p for p in candidates if p.exists()), None)
    if lulc_path is None:
        print(f"⚠️ LULC для {year} не знайдено")
        continue

    with rasterio.open(lulc_path) as src:
        assert src.crs.to_epsg() == 32635, f"❌ LULC CRS не EPSG:32635: {src.crs}"

        coords = [(geom.x, geom.y) for geom in subset.geometry]
        sampled = list(src.sample(coords))
        sampled_classes = [val[0] if val and len(val) > 0 else np.nan for val in sampled]


        if len(sampled_classes) != len(subset):
            print(f"⚠️ {year}: довжина даних не збігається — пропущено")
            continue

        # Додаємо поля
        subset["lulc_class"] = sampled_classes
        subset["lulc_name"] = [
            lulc_classes_colors.get(int(c), {}).get("name") if not np.isnan(c) else None
            for c in sampled_classes
        ]

        print(f"✅ {year}: оброблено {len(subset)} точок")

        gdfs.append(subset)

# 🧩 Об'єднуємо все в один GeoDataFrame
result_gdf = pd.concat(gdfs).sort_index()
result_gdf = gpd.GeoDataFrame(result_gdf, geometry="geometry", crs=ice_gdf.crs)

# 💾 Зберігаємо
result_path = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_lulc.parquet")
result_gdf.to_parquet(result_path)
print(f"\n📦 Успішно збережено: {result_path}")




✅ 2018: оброблено 1031077 точок
✅ 2019: оброблено 490663 точок
✅ 2020: оброблено 1313539 точок
✅ 2021: оброблено 1804071 точок
✅ 2022: оброблено 1441274 точок
✅ 2023: оброблено 320656 точок
✅ 2024: оброблено 568926 точок
✅ 2025: оброблено 341877 точок

📦 Успішно збережено: /mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_lulc.parquet


In [10]:
result_gdf


,region,sc_orient,track,segment_dist,solar_elevation,segment_id,background_rate,cycle,pair,rgt,...,abs_delta_nasa_dem,h_srtm_dem,delta_srtm_dem,abs_delta_srtm_dem,h_tan_dem,delta_tan_dem,abs_delta_tan_dem,year,lulc_class,lulc_name
2018-11-04 01:05:31.626116096,6.0,1.0,1.0,1.472979e+07,-40.574768,735401.0,2756.054548,1.0,0.0,556.0,...,17.270116,1015.0,-21.577062,21.577062,1026.908081,-9.668980,9.668980,2018,2.0,Trees
2018-11-04 01:05:31.626116096,6.0,1.0,1.0,1.472979e+07,-40.574768,735401.0,2756.054548,1.0,0.0,556.0,...,6.770848,1015.0,-11.077794,11.077794,1026.908081,0.830287,0.830287,2018,2.0,Trees
2018-11-04 01:05:31.626216192,6.0,1.0,1.0,1.472979e+07,-40.574768,735401.0,2756.054548,1.0,0.0,556.0,...,15.536839,1015.0,-19.843785,19.843785,1026.908081,-7.935704,7.935704,2018,2.0,Trees
2018-11-04 01:05:31.627616256,6.0,1.0,1.0,1.472979e+07,-40.574768,735401.0,2756.054548,1.0,0.0,556.0,...,15.088475,1015.0,-19.395421,19.395421,1026.908081,-7.487340,7.487340,2018,2.0,Trees
2018-11-04 01:05:31.629816064,6.0,1.0,1.0,1.472979e+07,-40.574768,735401.0,2756.054548,1.0,0.0,556.0,...,6.355321,1015.0,-10.662267,10.662267,1026.908081,1.245814,1.245814,2018,2.0,Trees
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-11 00:50:47.622379008,2.0,0.0,3.0,5.350995e+06,-51.285336,266853.0,11997.292335,26.0,1.0,396.0,...,14.834862,805.0,10.189415,10.189415,808.484314,13.673729,13.673729,2025,2.0,Trees
2025-01-11 00:50:47.622478848,2.0,0.0,3.0,5.350995e+06,-51.285336,266853.0,11997.292335,26.0,1.0,396.0,...,20.520409,805.0,15.874962,15.874962,808.484314,19.359276,19.359276,2025,2.0,Trees
2025-01-11 00:50:47.622579200,2.0,0.0,3.0,5.350995e+06,-51.285336,266853.0,11997.292335,26.0,1.0,396.0,...,8.165871,805.0,-12.811317,12.811317,808.484314,-9.327003,9.327003,2025,2.0,Trees
2025-01-11 00:50:47.622878976,2.0,0.0,3.0,5.350995e+06,-51.285336,266853.0,11997.292335,26.0,1.0,396.0,...,19.599938,805.0,14.954491,14.954491,808.484314,18.438805,18.438805,2025,2.0,Trees
